# Lab 11: 1D CNN for Time Series Forecasting

**University of Engineering and Technology Peshawar, Nowshera Campus**

**Course:** Machine Learning Lab

**Student Name:** Muhammad Ayub  
**Registration Number:** 22jzele0470

**Date:** May 14, 2026

---

## Table of Contents

1. [Introduction](#1-introduction)  
2. [Import Libraries](#2-import-libraries)  
3. [Model Parameters](#3-model-parameters)  
4. [Define 1D CNN Model](#4-define-1d-cnn-model)  
5. [Model Summary](#5-model-summary)  
6. [Callbacks Setup](#6-callbacks-setup)  
7. [Load and Prepare Dataset](#7-load-and-prepare-dataset)  
8. [Data Splitting (Univariate Multi-step)](#8-data-splitting-univariate-multi-step)  
9. [Model Training - Phase 1](#9-model-training-phase-1)  
10. [Model Evaluation - Phase 1](#10-model-evaluation-phase-1)  
11. [Model Training - Phase 2 (Fine-tuning)](#11-model-training-phase-2-fine-tuning)  
12. [Model Evaluation - Phase 2](#12-model-evaluation-phase-2)  

---

**GitHub Repository:**  
https://github.com/prince4775/8th-Semester-ML-and-DL-Lab

## 1. Introduction

In this lab, we implement a 1D Convolutional Neural Network (CNN) for univariate multi-step time series forecasting on the AEP hourly energy dataset.

## 2. Import Libraries

In [ ]:
import os
os.chdir(r'C:\Users\M Ayub\Downloads\ML_LAB')

from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

## 3. Model Parameters

In [ ]:
model = None
start_epoch = 0
time_steps = 24
num_features = 21

## 4. Define 1D CNN Model

In [ ]:
def CNN():
    input_data = Input(shape=(time_steps, num_features))
    x1 = Conv1D(16, 2, activation="relu")(input_data)
    x2 = Conv1D(16, 2, activation="relu")(x1)
    flatten = Flatten()(x2)
    output_data = Dense(1)(flatten)
    model = Model(input_data, output_data)
    return model

## 5. Model Summary

In [ ]:
model1 = CNN()
model1.summary()

## 6. Callbacks Setup

In [ ]:
checkpoints = r'C:\Users\M Ayub\Downloads\ML_LAB\lab_11\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\M Ayub\Downloads\ML_LAB\lab_11'
FIG_PATH = os.path.sep.join([OUTPUT_PATH, "history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH, "history.json"])

EpochCheckpoint1 = ModelCheckpoint(checkpoints, monitor="val_loss", save_best_only=True, verbose=1)
TrainingMonitor1 = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)
callbacks = [EpochCheckpoint1, TrainingMonitor1]

## 7. Load and Prepare Dataset

In [ ]:
path_dataset = r'C:\Users\M Ayub\Downloads\ML_LAB'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values

path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 

path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

## 8. Data Splitting (Univariate Multi-step)

In [ ]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0, target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0, target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0, target_len=1)
print('Time Consumed', time.time()-start, "sec")

## 9. Model Training - Phase 1

In [ ]:
if model is None:
    print("[INFO] compiling model...")
    model = CNN()
    opt = Adam(1e-3)
    model.compile(loss='mae', optimizer=opt, metrics=["mae", "mape"])

epochs = 60
verbose = 1
batch_size = 32

History = model.fit(train_X, train_y,
                    batch_size=batch_size,
                    epochs=epochs,
                    validation_data=(validation_X, validation_y),
                    callbacks=callbacks,
                    verbose=verbose)

## 10. Model Evaluation - Phase 1

In [ ]:
model = load_model(r'C:\Users\M Ayub\Downloads\ML_LAB\lab_11\E1-cp-0035-loss0.02.h5')

y_pred_scaled = model.predict(test_X)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)

MAE = np.mean(abs(y_pred - y_test_unscaled))
print('Mean Absolute Error (MAE):', np.round(MAE, 2))

MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE):', np.round(MEDAE, 2))

MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE):', np.round(MSE, 2))

RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE):', np.round(RMSE, 2))

MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred) / y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE):', np.round(MAPE, 2), '%')

MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred) / y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE):', np.round(MDAPE, 2), '%')

## 11. Model Training - Phase 2 (Fine-tuning)

In [ ]:
checkpoints = r'C:\Users\M Ayub\Downloads\ML_LAB\lab_11\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model = r'C:\Users\M Ayub\Downloads\ML_LAB\lab_11\E1-cp-0003-loss0.06.h5'
start_epoch = 58

EpochCheckpoint1 = ModelCheckpoint(checkpoints, monitor="val_loss", save_best_only=True, verbose=1)
TrainingMonitor1 = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)
callbacks = [EpochCheckpoint1, TrainingMonitor1]

model = load_model(model)
print("[INFO] old learning rate:", K.get_value(model.optimizer.lr))
K.set_value(model.optimizer.lr, 1e-4)
print("[INFO] new learning rate:", K.get_value(model.optimizer.lr))

epochs = 10
History = model.fit(train_X, train_y,
                    batch_size=32,
                    epochs=epochs,
                    validation_data=(validation_X, validation_y),
                    callbacks=callbacks,
                    verbose=1)

## 12. Model Evaluation - Phase 2

In [ ]:
model = load_model(r'C:\Users\M Ayub\Downloads\ML_LAB\lab_11\E1-cp-0035-loss0.02.h5')

y_pred_scaled = model.predict(test_X)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)

MAE = np.mean(abs(y_pred - y_test_unscaled))
print('Mean Absolute Error (MAE):', np.round(MAE, 2))

MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE):', np.round(MEDAE, 2))

MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE):', np.round(MSE, 2))

RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE):', np.round(RMSE, 2))

MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred) / y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE):', np.round(MAPE, 2), '%')

MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred) / y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE):', np.round(MDAPE, 2), '%')